# Step 9 - Train Baseline PPO (MaskablePPO + MultiInputPolicy)

This notebook implements Step 9 from the plan using the environment from Step 8.

Goals:
- Train MaskablePPO with MultiInputPolicy
- Use municipality hold-out experiments (A/B rotation)
- Run synthetic stress evaluation (C)
- Compare RL policy vs heuristic baselines
- Export `ppo_metrics.csv` and `ppo_checkpoint/` artifacts

In [ ]:
%pip install stable-baselines3 sb3-contrib nbformat

In [27]:
import json
import time
from datetime import datetime
from pathlib import Path
from typing import Callable, Dict, List

import gymnasium as gym
import nbformat
import numpy as np
import pandas as pd
from stable_baselines3.common.vec_env import DummyVecEnv
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import get_action_masks

ROOT = Path('.')
OUTPUT_DIR = ROOT / 'output'
STEP8_NOTEBOOK = ROOT / 'step8_gymnasium_environment.ipynb'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PPO_METRICS_PATH = OUTPUT_DIR / 'ppo_metrics.csv'
PPO_CHECKPOINT_DIR = OUTPUT_DIR / 'ppo_checkpoint'
PPO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUTPUT_DIR.resolve())
print('Step 8 notebook found:', STEP8_NOTEBOOK.exists())
print('Metrics file target:', PPO_METRICS_PATH)

Output dir: C:\Users\Lenovo\Documents\React Projects\bureaucratic-workflow-analyzer\output
Step 8 notebook found: True
Metrics file target: output\ppo_metrics.csv


## Load Step 8 Core Objects

To keep Step 9 aligned with Step 8, this cell executes only the required Step 8 code sections:
- artifact/path setup
- artifact loading
- lookup builders
- `Case` dataclass
- `BPICMultiCaseEnv` class

In [28]:
def _cell_text(cell):
    src = cell.get('source', '')
    if isinstance(src, list):
        return '\n'.join(src)
    return str(src)


def exec_step8_upto_env(nb_path: Path):
    """Execute Step 8 code cells in order up to and including env definition."""
    with nb_path.open('r', encoding='utf-8') as f:
        nb = json.load(f)

    executed_cells = 0
    found_env_cell = False

    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue

        code = _cell_text(cell)

        # Skip notebook package install cells.
        lines = []
        for line in code.splitlines():
            if line.strip().startswith('%pip'):
                continue
            lines.append(line)
        code = '\n'.join(lines).strip()

        if not code:
            continue

        exec(code, globals())
        executed_cells += 1

        if 'class BPICMultiCaseEnv(gym.Env):' in code:
            found_env_cell = True
            break

    if not found_env_cell:
        raise RuntimeError('Could not find BPICMultiCaseEnv definition while loading Step 8.')

    return executed_cells


executed = exec_step8_upto_env(STEP8_NOTEBOOK)

# Explicitly bind dynamic symbols so static analyzers can resolve names.
BPICMultiCaseEnv = globals()['BPICMultiCaseEnv']
Case = globals()['Case']
resource_config = globals()['resource_config']
transition_lookup = globals()['transition_lookup']
duration_lookup = globals()['duration_lookup']
features_df = globals()['features_df']

required_names = [
    'Case',
    'BPICMultiCaseEnv',
    'resource_config',
    'transition_lookup',
    'duration_lookup',
    'features_df',
]
for name in required_names:
    assert name in globals(), f'Missing required object: {name}'

print(f'Loaded Step 8 objects successfully (executed {executed} code cells)')
print('Municipality configs available:', sorted(list(resource_config.get('by_municipality', {}).keys())))

Output dir: C:\Users\Lenovo\Documents\React Projects\bureaucratic-workflow-analyzer\output
Checking required files...
  case_step_features.parquet: ✓
  graph_priors.json: ✓
  transition_stats.csv: ✓
  duration_stats.csv: ✓
  valid_action_space.csv: ✓
  sim_calibration.json: ✓
  resource_calibration.json: ✓
Features table: 262,628 rows
Activity ranks: 5 municipalities
Action space: 15 actions
Resource config loaded: Little's Law + Work-in-Progress analysis from BPIC15 logs
Reward params loaded: alpha=0.021521, gamma=12.000000

RESOURCE CALIBRATION FROM STEP 3.5

Method: Little's Law + Work-in-Progress analysis from BPIC15 logs
Description: Realistic workforce requirements calibrated from concurrent case analysis

Worker Pool by Municipality (calibrated from BPIC15 logs):
Munic.   Peak Cases      Min Staff       Recommended     Max Staff      
--------------------------------------------------------------------
M1       139             6               7               10             
M2  

In [ ]:
class DictScalarToVectorObsWrapper(gym.ObservationWrapper):
    """Convert scalar Dict observation entries (shape=()) to shape=(1,) for SB3 MultiInputPolicy."""

    def __init__(self, env):
        super().__init__(env)
        old_space = env.observation_space
        if not isinstance(old_space, gym.spaces.Dict):
            raise TypeError('Expected Dict observation space')

        self.scalar_keys = []
        new_spaces = {}

        for key, space in old_space.spaces.items():
            if isinstance(space, gym.spaces.Box) and space.shape == ():
                low = np.array([space.low], dtype=space.dtype)
                high = np.array([space.high], dtype=space.dtype)
                new_spaces[key] = gym.spaces.Box(low=low, high=high, shape=(1,), dtype=space.dtype)
                self.scalar_keys.append(key)
            else:
                new_spaces[key] = space

        self.observation_space = gym.spaces.Dict(new_spaces)

    def observation(self, observation):
        obs = dict(observation)
        for key in self.scalar_keys:
            target_dtype = self.observation_space[key].dtype
            obs[key] = np.asarray(obs[key], dtype=target_dtype).reshape(1,)
        return obs

    def action_masks(self):
        """Forward action masking call to wrapped base environment."""
        return self.env.action_masks()


class RewardBonusScaleWrapper(gym.Wrapper):
    """Rescale action bonus contribution to mitigate reward hacking in Step 9 training/eval."""

    def __init__(self, env, bonus_scale: float = 1.0):
        super().__init__(env)
        self.bonus_scale = float(bonus_scale)

    def action_masks(self):
        return self.env.action_masks()

    def step(self, action):
        obs, reward, done, truncated, info = self.env.step(action)

        if isinstance(info, dict):
            rc = info.get('reward_components', {})
            if isinstance(rc, dict):
                original_bonus = float(rc.get('action_bonus', 0.0))
                scaled_bonus = original_bonus * self.bonus_scale
                reward = float(reward) - original_bonus + scaled_bonus
                rc['action_bonus_original'] = original_bonus
                rc['action_bonus'] = scaled_bonus
                info['reward_components'] = rc

        return obs, reward, done, truncated, info


def action_mask_fn(env):
    return env.action_masks()


def make_masked_env(
    municipality: int,
    seed: int,
    arrival_rate: float,
    max_episode_hours: int = 168,
    bonus_scale: float = 1.0,
):
    env = BPICMultiCaseEnv(
        municipality=municipality,
        seed=seed,
        max_episode_hours=max_episode_hours,
        resource_config=resource_config,
    )
    env.arrival_rate = arrival_rate

    # Enforce Step 9 simulator guardrails even if a stale Step 8 class is loaded.
    if hasattr(env, 'reward_mode'):
        env.reward_mode = 'kpi_tuned'
    if hasattr(env, 'sim_time_scaling'):
        env.sim_time_scaling = 1.0
    if hasattr(env, 'reward_params') and isinstance(env.reward_params, dict):
        env.reward_params['gamma'] = max(float(env.reward_params.get('gamma', 12.0)), 40.0)

    # Reduce reward hacking by scaling action bonuses during Step 9 experiments.
    env = RewardBonusScaleWrapper(env, bonus_scale=bonus_scale)

    # SB3 MultiInputPolicy expects each Dict observation entry to have at least 1 dimension.
    env = DictScalarToVectorObsWrapper(env)
    return ActionMasker(env, action_mask_fn)


def make_vec_env(municipalities: List[int], seed: int, arrival_rate: float, bonus_scale: float = 1.0):
    env_fns = []
    for i, m in enumerate(municipalities):
        local_seed = seed + i * 17
        env_fns.append(lambda m=m, s=local_seed: make_masked_env(m, s, arrival_rate, bonus_scale=bonus_scale))
    return DummyVecEnv(env_fns)


# Smoke-check mask and observation validity
env_check = make_masked_env(municipality=1, seed=42, arrival_rate=2.0, bonus_scale=0.0)
obs, info = env_check.reset()
mask = env_check.action_masks()
print('Action mask length:', len(mask), 'Valid actions:', int(mask.sum()))
print('Obs shapes:', {k: np.asarray(v).shape for k, v in obs.items()})
assert mask.sum() >= 1, 'Every state must have at least one valid action'

Action mask length: 15 Valid actions: 14
Obs shapes: {'queue_lengths': (15,), 'active_case_ages': (100,), 'available_workers': (1,), 'current_hour': (1,)}


## Training and Evaluation Helpers

In [30]:
def _unwrap_for_action_masks(env):
    """Best-effort unwrap to an object exposing action_masks()."""
    current = env
    visited = set()

    for _ in range(10):
        if hasattr(current, 'action_masks'):
            return current

        key = id(current)
        if key in visited:
            break
        visited.add(key)

        if hasattr(current, 'env'):
            current = current.env
            continue

        if hasattr(current, 'venv'):
            current = current.venv
            continue

        if hasattr(current, 'envs') and len(current.envs) > 0:
            current = current.envs[0]
            continue

        break

    raise AttributeError('Could not find action_masks() on wrapped environment')


def _unwrap_to_stateful_env(env):
    """Best-effort unwrap to env exposing active_cases / total_workers / completed_cases."""
    current = env
    visited = set()

    for _ in range(12):
        if all(hasattr(current, attr) for attr in ['active_cases', 'total_workers', 'completed_cases']):
            return current

        key = id(current)
        if key in visited:
            break
        visited.add(key)

        if hasattr(current, 'env'):
            current = current.env
            continue

        if hasattr(current, 'venv'):
            current = current.venv
            continue

        if hasattr(current, 'envs') and len(current.envs) > 0:
            current = current.envs[0]
            continue

        break

    raise AttributeError('Could not unwrap to base env with case-level state attributes')


def heuristic_policy_action(env, mode='fcfs'):
    base_env = _unwrap_for_action_masks(env)
    mask = base_env.action_masks()
    valid = np.where(mask == 1)[0]
    if len(valid) == 0:
        return 2

    if mode == 'fcfs':
        if 2 in valid:
            return 2
    elif mode == 'throughput':
        if 10 in valid:
            return 10
    elif mode == 'close_when_possible':
        if 14 in valid:
            return 14

    return int(valid[0])


def run_eval_episode(env, policy_mode='rl', model=None, heuristic_mode='fcfs'):
    obs, info = env.reset()
    total_reward = 0.0
    steps = 0
    done = False
    truncated = False

    while not (done or truncated):
        if policy_mode == 'rl':
            masks = get_action_masks(env)
            action, _ = model.predict(obs, action_masks=masks, deterministic=True)
        else:
            action = heuristic_policy_action(env, mode=heuristic_mode)

        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward)
        steps += 1

    return {
        'reward': total_reward,
        'steps': steps,
        'completed_cases': info.get('completed_cases', 0),
        'queue_length': info.get('queue_length', 0),
        'workers': info.get('total_workers', 0),
    }


def evaluate_policy_on_municipality(
    model,
    municipality: int,
    arrival_rate: float,
    episodes: int,
    seed: int,
    bonus_scale: float = 0.0,
):
    rows = []
    for ep in range(episodes):
        env = make_masked_env(municipality, seed + ep, arrival_rate, bonus_scale=bonus_scale)
        out = run_eval_episode(env, policy_mode='rl', model=model)
        rows.append(out)
    df = pd.DataFrame(rows)
    return {
        'avg_reward': float(df['reward'].mean()),
        'avg_completed': float(df['completed_cases'].mean()),
        'avg_queue': float(df['queue_length'].mean()),
        'avg_steps': float(df['steps'].mean()),
    }


def evaluate_heuristic_on_municipality(
    municipality: int,
    heuristic_mode: str,
    arrival_rate: float,
    episodes: int,
    seed: int,
    bonus_scale: float = 0.0,
):
    rows = []
    for ep in range(episodes):
        env = make_masked_env(municipality, seed + ep, arrival_rate, bonus_scale=bonus_scale)
        out = run_eval_episode(env, policy_mode='heuristic', heuristic_mode=heuristic_mode)
        rows.append(out)
    df = pd.DataFrame(rows)
    return {
        'avg_reward': float(df['reward'].mean()),
        'avg_completed': float(df['completed_cases'].mean()),
        'avg_queue': float(df['queue_length'].mean()),
        'avg_steps': float(df['steps'].mean()),
    }

## Experiment Matrix (A, B, C)

- A/B: municipality hold-out rotation
- C: stress evaluation with higher arrival rate

In [31]:
seed = 42

# Use 'smoke' for quick pipeline checks; use 'standard' for meaningful training signal.
training_profile = 'standard'

if training_profile == 'smoke':
    total_timesteps = 30000
    n_steps = 512
    batch_size = 128
    eval_episodes = 6
else:
    total_timesteps = 500000
    n_steps = 2048
    batch_size = 256
    eval_episodes = 12

# Train on elevated load so completion signal is frequent enough.
train_arrival_rate = 12.0

# Evaluate at realistic load, and keep stress scenario separate.
eval_arrival_rate = 2.0
stress_arrival_rate = 12.0

# Neutralize action-bonus hacking in Step 9 runs (use KPI-driven reward signal).
train_bonus_scale = 0.0
eval_bonus_scale = 0.0

experiments = []
for held_out in [1, 2, 3, 4, 5]:
    train_municipalities = [m for m in [1, 2, 3, 4, 5] if m != held_out]
    experiments.append({
        'name': f'holdout_M{held_out}',
        'train_municipalities': train_municipalities,
        'held_out': held_out,
    })

print('Training profile:', training_profile)
print('Timesteps:', total_timesteps, '| n_steps:', n_steps, '| batch_size:', batch_size)
print('Arrival rates -> train:', train_arrival_rate, '/ eval:', eval_arrival_rate, '/ stress:', stress_arrival_rate)
print('Bonus scales -> train:', train_bonus_scale, '/ eval:', eval_bonus_scale)
print('Planned experiments:')
for exp in experiments:
    print(exp)

Training profile: standard
Timesteps: 500000 | n_steps: 2048 | batch_size: 256
Arrival rates -> train: 12.0 / eval: 2.0 / stress: 12.0
Bonus scales -> train: 0.0 / eval: 0.0
Planned experiments:
{'name': 'holdout_M1', 'train_municipalities': [2, 3, 4, 5], 'held_out': 1}
{'name': 'holdout_M2', 'train_municipalities': [1, 3, 4, 5], 'held_out': 2}
{'name': 'holdout_M3', 'train_municipalities': [1, 2, 4, 5], 'held_out': 3}
{'name': 'holdout_M4', 'train_municipalities': [1, 2, 3, 5], 'held_out': 4}
{'name': 'holdout_M5', 'train_municipalities': [1, 2, 3, 4], 'held_out': 5}


In [ ]:
# Lightweight quick-validation loop with action-level logs
quick_train_timesteps = 100000
quick_eval_steps = 80
quick_seed = seed + 999
quick_municipality = 1
quick_train_municipalities = [1, 2, 3, 4, 5]
quick_eval_arrival_rate = eval_arrival_rate
quick_behavior_arrival_rate = max(eval_arrival_rate, 4.0)

print('=' * 90)
print('QUICK VALIDATION: train once, evaluate deterministic + stochastic + behavior load')
print('=' * 90)
print(f'Municipality: M{quick_municipality} | timesteps={quick_train_timesteps} | eval_steps={quick_eval_steps}')
print(f'Quick-train municipalities: {quick_train_municipalities}')
print(f'Using train_arrival_rate={train_arrival_rate}, eval_arrival_rate={quick_eval_arrival_rate}, behavior_arrival_rate={quick_behavior_arrival_rate}')

# Mixed-load quick curriculum avoids single-load local optima.
quick_train_arrival_by_m = {1: 6.0, 2: 8.0, 3: 10.0, 4: 12.0, 5: 4.0}
env_fns = []
for i, m in enumerate(quick_train_municipalities):
    local_seed = quick_seed + i * 17
    local_rate = quick_train_arrival_by_m.get(m, train_arrival_rate)
    env_fns.append(lambda m=m, s=local_seed, r=local_rate: make_masked_env(m, s, r, bonus_scale=train_bonus_scale))
quick_train_env = DummyVecEnv(env_fns)

quick_model = MaskablePPO(
    policy='MultiInputPolicy',
    env=quick_train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=256,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.05,
    clip_range=0.2,
    verbose=0,
    seed=quick_seed,
)

start_t = time.time()
quick_model.learn(total_timesteps=quick_train_timesteps)
print(f'Quick training completed in {time.time() - start_t:.2f}s')

action_name_map = globals().get('ACTION_MAP', {})

def run_logged_rollout(label: str, arrival_rate: float, deterministic: bool, seed_offset: int = 0):
    print('-' * 90)
    print(f'ROLLOUT [{label}] | arrival_rate={arrival_rate} | deterministic={deterministic}')

    log_env = make_masked_env(
        quick_municipality,
        seed=quick_seed + 1 + seed_offset,
        arrival_rate=arrival_rate,
        bonus_scale=eval_bonus_scale,
    )
    obs, _ = log_env.reset()
    base_env = _unwrap_to_stateful_env(log_env)

    prev_queue = len(base_env.active_cases)
    prev_workers = base_env.total_workers
    prev_completed = len(base_env.completed_cases)

    action_counter = {}
    rows = []

    for t in range(1, quick_eval_steps + 1):
        masks = get_action_masks(log_env)
        action, _ = quick_model.predict(obs, action_masks=masks, deterministic=deterministic)
        action_id = int(action)
        action_name = action_name_map.get(action_id, f'action_{action_id}')
        action_counter[action_id] = action_counter.get(action_id, 0) + 1

        obs, reward, done, truncated, info = log_env.step(action_id)
        rc = info.get('reward_components', {}) if isinstance(info, dict) else {}
        comp_r = float(rc.get('completion', 0.0)) if isinstance(rc, dict) else 0.0
        queue_r = float(rc.get('queue_penalty', 0.0)) if isinstance(rc, dict) else 0.0
        sla_r = float(rc.get('sla_penalty', 0.0)) if isinstance(rc, dict) else 0.0

        curr_queue = int(info.get('queue_length', 0))
        curr_workers = int(info.get('total_workers', 0))
        curr_completed = int(info.get('completed_cases', 0))

        d_queue = curr_queue - prev_queue
        d_workers = curr_workers - prev_workers
        d_completed = curr_completed - prev_completed
        valid_actions = np.where(masks == 1)[0].tolist()

        row = {
            'rollout': label,
            'step': t,
            'action_id': action_id,
            'action_name': action_name,
            'reward': float(reward),
            'queue': curr_queue,
            'd_queue': d_queue,
            'workers': curr_workers,
            'd_workers': d_workers,
            'completed': curr_completed,
            'd_completed': d_completed,
            'completion_reward': comp_r,
            'queue_penalty': queue_r,
            'sla_penalty': sla_r,
            'valid_action_count': len(valid_actions),
        }
        rows.append(row)

        print(
            f"t={t:03d} | action={action_id:02d} ({action_name}) | "
            f"reward={float(reward):+0.3f} [c={comp_r:+0.3f}, q={queue_r:+0.3f}, sla={sla_r:+0.3f}] | "
            f"queue={curr_queue:3d} ({d_queue:+d}) | workers={curr_workers:2d} ({d_workers:+d}) | "
            f"completed={curr_completed:3d} ({d_completed:+d})"
        )

        prev_queue = curr_queue
        prev_workers = curr_workers
        prev_completed = curr_completed

        if done or truncated:
            print(f'Episode ended early at step {t} (done={done}, truncated={truncated})')
            break

    df = pd.DataFrame(rows)
    path = OUTPUT_DIR / f'quick_validation_{label}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df.to_csv(path, index=False)

    print('Summary:')
    print(f"  Steps: {len(df)}")
    print(f"  Total reward: {df['reward'].sum():.3f}")
    print(f"  Final queue: {int(df['queue'].iloc[-1]) if len(df) else 0}")
    print(f"  Final completed: {int(df['completed'].iloc[-1]) if len(df) else 0}")

    diagnostics = {'dominant': 1.0, 'effective_actions': 1, 'total_sla': 0.0, 'total_queue': 0.0, 'total_comp': 0.0}
    if len(df) > 0:
        action_dist = pd.Series(action_counter).sort_index()
        action_dist = action_dist / action_dist.sum()
        print('  Action distribution:')
        for aid, frac in action_dist.items():
            aname = action_name_map.get(int(aid), f'action_{int(aid)}')
            print(f'    action {int(aid):02d} ({aname}): {frac:.1%}')

        dominant = float(action_dist.max()) if len(action_dist) > 0 else 0.0
        effective_actions = int((action_dist > 0).sum())
        total_comp = float(df['completion_reward'].sum())
        total_queue = float(df['queue_penalty'].sum())
        total_sla = float(df['sla_penalty'].sum())
        diagnostics.update({
            'dominant': dominant,
            'effective_actions': effective_actions,
            'total_sla': total_sla,
            'total_queue': total_queue,
            'total_comp': total_comp,
        })

        if dominant >= 0.7 or effective_actions <= 3:
            print('  WARNING: action concentration is high.')
        print(f'  Reward totals -> completion={total_comp:+.3f}, queue={total_queue:+.3f}, sla={total_sla:+.3f}')
        if abs(total_sla) > (1.5 * abs(total_queue) + abs(total_comp)):
            print('  WARNING: SLA penalty dominates return. Queue is aging faster than completions.')

    print(f'  Detailed log saved to: {path}')
    return {'label': label, 'path': str(path), 'df': df, 'diagnostics': diagnostics}

res_det = run_logged_rollout('det_eval', quick_eval_arrival_rate, deterministic=True, seed_offset=0)
res_sto = run_logged_rollout('sto_eval', quick_eval_arrival_rate, deterministic=False, seed_offset=100)
res_beh = run_logged_rollout('det_behavior_load', quick_behavior_arrival_rate, deterministic=True, seed_offset=200)

print('-' * 90)
print('Quick validation gate:')
det_diag = res_det['diagnostics']
det_pass = (
    (det_diag['dominant'] < 0.85)
    and (det_diag['effective_actions'] >= 3)
    and (det_diag['total_comp'] > 0.0)
    and (abs(det_diag['total_sla']) <= (2.0 * abs(det_diag['total_queue']) + abs(det_diag['total_comp'])))
)
print(f"  det_eval dominant={det_diag['dominant']:.1%}, effective_actions={det_diag['effective_actions']}, completion_total={det_diag['total_comp']:+.3f}, sla_total={det_diag['total_sla']:+.3f}")
print(f'  PASS={det_pass}')
if not det_pass:
    print('  NOTE: Model not ready for heavy loop yet. Increase timesteps or tune entropy/learning settings.')

print('=' * 90)
print('Quick validation finished.')
print('Use det_eval for trustable policy behavior, sto_eval for collapse diagnostics, det_behavior_load for workload robustness.')
print('=' * 90)

QUICK VALIDATION: train once, evaluate deterministic + stochastic + behavior load
Municipality: M1 | timesteps=100000 | eval_steps=80
Quick-train municipalities: [1, 2, 3, 4, 5]
Using train_arrival_rate=12.0, eval_arrival_rate=2.0, behavior_arrival_rate=4.0
Quick training completed in 100.76s
------------------------------------------------------------------------------------------
ROLLOUT [det_eval] | arrival_rate=2.0 | deterministic=True
t=001 | action=10 (enable_cross_trained_pool) | reward=+0.000 [c=+0.000, q=-0.000, sla=-0.000] | queue=  0 (+0) | workers= 7 (+0) | completed=  0 (+0)
t=002 | action=10 (enable_cross_trained_pool) | reward=+0.000 [c=+0.000, q=-0.000, sla=-0.000] | queue=  0 (+0) | workers= 7 (+0) | completed=  0 (+0)
t=003 | action=10 (enable_cross_trained_pool) | reward=+0.000 [c=+0.000, q=-0.000, sla=-0.000] | queue=  0 (+0) | workers= 7 (+0) | completed=  0 (+0)
t=004 | action=10 (enable_cross_trained_pool) | reward=+0.000 [c=+0.000, q=-0.000, sla=-0.000] | queue=

## Stop Point Before Heavy Training

You can run Cells 1 to 11 for setup plus quick validation.

If quick validation looks good, continue to the next cell for the full hold-out experiment loop (computationally heavy).

In [ ]:
results = []
run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')

for i, exp in enumerate(experiments):
    exp_name = exp['name']
    train_muns = exp['train_municipalities']
    held_out = exp['held_out']

    print('\n' + '=' * 90)
    print(f"Experiment {i+1}/{len(experiments)}: {exp_name}")
    print(f"Train municipalities: {train_muns} | Held out: M{held_out}")
    print('=' * 90)

    train_env = make_vec_env(
        train_muns,
        seed=seed + i * 100,
        arrival_rate=train_arrival_rate,
        bonus_scale=train_bonus_scale,
    )

    model = MaskablePPO(
        policy='MultiInputPolicy',
        env=train_env,
        learning_rate=3e-4,
        n_steps=n_steps,
        batch_size=batch_size,
        gamma=0.99,
        gae_lambda=0.95,
        ent_coef=0.01,
        clip_range=0.2,
        verbose=0,
        seed=seed + i,
    )

    t0 = time.time()
    model.learn(total_timesteps=total_timesteps)
    train_seconds = time.time() - t0

    ckpt_path = PPO_CHECKPOINT_DIR / f'{run_tag}_{exp_name}.zip'
    model.save(str(ckpt_path))

    # RL evaluation on held-out municipality
    rl_eval = evaluate_policy_on_municipality(
        model=model,
        municipality=held_out,
        arrival_rate=eval_arrival_rate,
        episodes=eval_episodes,
        seed=seed + 1000 + i * 10,
        bonus_scale=eval_bonus_scale,
    )

    # Stress evaluation (Exp C flavor)
    rl_stress = evaluate_policy_on_municipality(
        model=model,
        municipality=held_out,
        arrival_rate=stress_arrival_rate,
        episodes=eval_episodes,
        seed=seed + 2000 + i * 10,
        bonus_scale=eval_bonus_scale,
    )

    # Heuristic baselines
    h_fcfs = evaluate_heuristic_on_municipality(
        municipality=held_out,
        heuristic_mode='fcfs',
        arrival_rate=eval_arrival_rate,
        episodes=eval_episodes,
        seed=seed + 3000 + i * 10,
        bonus_scale=eval_bonus_scale,
    )

    h_throughput = evaluate_heuristic_on_municipality(
        municipality=held_out,
        heuristic_mode='throughput',
        arrival_rate=eval_arrival_rate,
        episodes=eval_episodes,
        seed=seed + 4000 + i * 10,
        bonus_scale=eval_bonus_scale,
    )

    row = {
        'run_tag': run_tag,
        'experiment': exp_name,
        'held_out_municipality': held_out,
        'train_municipalities': ','.join(map(str, train_muns)),
        'timesteps': total_timesteps,
        'n_steps': n_steps,
        'batch_size': batch_size,
        'train_arrival_rate': train_arrival_rate,
        'eval_arrival_rate': eval_arrival_rate,
        'train_bonus_scale': train_bonus_scale,
        'eval_bonus_scale': eval_bonus_scale,
        'train_seconds': round(train_seconds, 2),
        'checkpoint_path': str(ckpt_path),

        'rl_avg_reward': rl_eval['avg_reward'],
        'rl_avg_completed': rl_eval['avg_completed'],
        'rl_avg_queue': rl_eval['avg_queue'],

        'rl_stress_avg_reward': rl_stress['avg_reward'],
        'rl_stress_avg_completed': rl_stress['avg_completed'],
        'rl_stress_avg_queue': rl_stress['avg_queue'],

        'heur_fcfs_avg_reward': h_fcfs['avg_reward'],
        'heur_fcfs_avg_completed': h_fcfs['avg_completed'],
        'heur_fcfs_avg_queue': h_fcfs['avg_queue'],

        'heur_throughput_avg_reward': h_throughput['avg_reward'],
        'heur_throughput_avg_completed': h_throughput['avg_completed'],
        'heur_throughput_avg_queue': h_throughput['avg_queue'],
    }
    results.append(row)

    print('Checkpoint saved:', ckpt_path)
    print(f"RL eval reward={row['rl_avg_reward']:.3f}, completed={row['rl_avg_completed']:.2f}, queue={row['rl_avg_queue']:.2f}")
    print(f"Heuristic FCFS reward={row['heur_fcfs_avg_reward']:.3f}, completed={row['heur_fcfs_avg_completed']:.2f}")

results_df = pd.DataFrame(results)
results_df.to_csv(PPO_METRICS_PATH, index=False)

print('\nSaved metrics to:', PPO_METRICS_PATH)
display(results_df)

In [ ]:
if 'results_df' not in globals() or results_df is None or len(results_df) == 0:
    print('No full-experiment results found yet. Stop point respected (quick-validation only).')
    print('Run the heavy full experiment loop cell first if you want this summary table.')
else:
    summary_cols = [
        'experiment',
        'held_out_municipality',
        'rl_avg_reward',
        'heur_fcfs_avg_reward',
        'heur_throughput_avg_reward',
        'rl_stress_avg_reward',
    ]
    summary = results_df[summary_cols].copy()
    summary['rl_minus_fcfs'] = summary['rl_avg_reward'] - summary['heur_fcfs_avg_reward']
    summary['rl_minus_throughput'] = summary['rl_avg_reward'] - summary['heur_throughput_avg_reward']
    print('Evaluation summary:')
    display(summary)

    wins_fcfs = int((summary['rl_minus_fcfs'] > 0).sum())
    wins_tp = int((summary['rl_minus_throughput'] > 0).sum())
    print(f'RL wins vs FCFS in {wins_fcfs}/{len(summary)} hold-outs')
    print(f'RL wins vs Throughput in {wins_tp}/{len(summary)} hold-outs')

## Step 9 Outputs

- `output/ppo_metrics.csv`
- `output/ppo_checkpoint/*.zip`

If training is too slow, reduce `total_timesteps` or reduce the number of hold-out experiments while iterating.